# Phase 7 — Avaliação completa do modelo finetunado

Gera 3600 imagens com SD 1.5 + LoRA (10 epochs), julga com VLM, e compara com baseline da Fase 3.

**Runtime total:** 6-8h em A100. Geração ~4-6h, julgamento ~1-2h, comparação ~1min.

**Idempotente:** disconnect não perde progresso, retoma onde parou.

## 1. Mount Drive e clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/GabrielmAlves/dgm-2026.1.git"
BRANCH = "feature/generate_datasets"
import os, subprocess
REPO_DIR = "/content/binding-research"
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR/projects/challenges-in-attribute-control

## 2. Install deps

In [ ]:
!pip install -q uv
!uv pip install -e . --system
!pip install -q diffusers accelerate peft transformers
!pip uninstall -y torchao

## 3. HF login

In [ ]:
from huggingface_hub import login
login()

## 4. Confirm A100

In [ ]:
import torch
assert torch.cuda.is_available()
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name} ({vram:.1f} GB)')
assert vram >= 35, 'A100 needed'

## 5. Restaurar LoRA salva (de 10 epochs)

A LoRA treinada por 10 epochs deve estar em `MyDrive/binding-research/lora_output_v1_10ep/`.

In [ ]:
import os, shutil
LORA_DRIVE = '/content/drive/MyDrive/binding-research/lora_output_v1_10ep'
assert os.path.exists(LORA_DRIVE), f'LoRA não encontrada em {LORA_DRIVE}'
if os.path.exists('/content/lora_output'):
    shutil.rmtree('/content/lora_output')
shutil.copytree(LORA_DRIVE, '/content/lora_output')
print('Restaurada. Conteúdo:', os.listdir('/content/lora_output'))

## 6. Geração das 3600 imagens com LoRA

**Este é o passo longo — ~4-6h em A100.** Salva incrementalmente no Drive a cada 120 imagens para resistir a disconnect.

In [ ]:
!python experiments/exp7_generate_lora_eval.py \
    --config configs/exp3_default.yaml \
    --lora-unet /content/lora_output/lora_unet \
    --lora-text /content/lora_output/lora_text_encoder \
    --out-dir /content/eval_images_lora \
    --checkpoint-every 120 \
    --drive-mirror /content/drive/MyDrive/binding-research/eval_images_lora

## 7. Julgamento VLM dos 3600 imagens

Reutiliza `exp3_judge.py` apontando para o novo manifest. ~1-2h em A100.

In [ ]:
!python experiments/exp3_judge.py \
    --manifest /content/eval_images_lora/manifest.csv \
    --config configs/judge_default.yaml \
    --out-csv /content/judgments_lora.csv

## 8. Análise comparativa baseline vs LoRA

In [ ]:
import shutil, os
BASELINE = '/content/drive/MyDrive/binding-research/data/judgments/judgments.csv'
NPMI = '/content/drive/MyDrive/binding-research/data/exp1_results/npmi_per_pair.csv'
SPLIT = '/content/drive/MyDrive/binding-research/exp5_split/finetuning_split.csv'
for p in [BASELINE, NPMI, SPLIT]:
    if not os.path.exists(p):
        print(f'AVISO: {p} não existe no Drive. Sobe via navegador antes deste passo.')

os.makedirs('data/judgments', exist_ok=True)
os.makedirs('data/exp1_results', exist_ok=True)
os.makedirs('results/exp5_split', exist_ok=True)
shutil.copy(BASELINE, 'data/judgments/judgments.csv')
shutil.copy(NPMI, 'data/exp1_results/npmi_per_pair.csv')
shutil.copy(SPLIT, 'results/exp5_split/finetuning_split.csv')
shutil.copy('/content/judgments_lora.csv', 'data/judgments/judgments_lora.csv')

In [ ]:
!python experiments/exp7_compare.py \
    --baseline-judgments data/judgments/judgments.csv \
    --lora-judgments data/judgments/judgments_lora.csv \
    --npmi-per-pair data/exp1_results/npmi_per_pair.csv \
    --split results/exp5_split/finetuning_split.csv \
    --out-dir results/exp7_compare

## 9. Visualizar resultado

In [ ]:
from PIL import Image
from IPython.display import display
import pandas as pd, json

print('=== Per-pair results ===')
df = pd.read_csv('results/exp7_compare/per_pair.csv')
print(df.sort_values('delta', ascending=False).head(20).to_string())
print()
print('=== Group summary ===')
with open('results/exp7_compare/group_summary.json') as f:
    s = json.load(f)
print(json.dumps(s, indent=2))
print()
display(Image.open('results/exp7_compare/per_pair_plot.png'))

## 10. Salvar resultados no Drive

In [ ]:
import shutil
DEST = '/content/drive/MyDrive/binding-research/results/exp7_compare'
shutil.copytree('results/exp7_compare', DEST, dirs_exist_ok=True)
shutil.copy('/content/judgments_lora.csv', '/content/drive/MyDrive/binding-research/data/judgments/judgments_lora.csv')
print(f'Salvo: {DEST}')